# 01 - Configuração do Projeto e Base Sintética

Este notebook configura a estrutura inicial do projeto **Agro Leads Orchestrator**.

Nesta etapa, serão criados:

- banco SQLite local;
- schema relacional;
- base sintética de leads agrícolas;
- índices SQL de performance;
- consultas iniciais de validação.

A proposta é simular uma operação real de uma Agritech com grande volume de leads comerciais.

## Objetivo da etapa

O objetivo deste notebook é preparar a camada inicial de dados do projeto.

A base criada será utilizada nas próximas etapas para:

- análise exploratória;
- engenharia de dados;
- orquestração por máquina de estados;
- simulação de operação comercial;
- modelos de Machine Learning;
- dashboards analíticos.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

In [2]:
RAIZ_PROJETO = Path.cwd()

if RAIZ_PROJETO.name == "notebooks":
    RAIZ_PROJETO = RAIZ_PROJETO.parent

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.append(str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)

Raiz do projeto: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator


In [3]:
#Importar módulos internos

from src.banco import (
    criar_conexao,
    remover_banco_existente,
    criar_schema,
    criar_indices
)

from src.gerador import (
    gerar_lote_leads,
    inserir_leads_em_lotes
)

In [4]:
#Configurações da carga

CAMINHO_DADOS = RAIZ_PROJETO / "dados"
CAMINHO_DADOS.mkdir(parents=True, exist_ok=True)

CAMINHO_BANCO = CAMINHO_DADOS / "agro_leads.db"

QUANTIDADE_LEADS = 50_000
TAMANHO_LOTE = 10_000
SEMENTE_ALEATORIA = 42

gerador = np.random.default_rng(SEMENTE_ALEATORIA)

print("Caminho do banco:", CAMINHO_BANCO)
print("Quantidade de leads:", QUANTIDADE_LEADS)
print("Tamanho do lote:", TAMANHO_LOTE)

Caminho do banco: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator\dados\agro_leads.db
Quantidade de leads: 50000
Tamanho do lote: 10000


In [5]:
#Gerar amostra de teste

amostra_leads = gerar_lote_leads(
    id_inicial=1,
    quantidade_lote=10,
    gerador=gerador
)

amostra_leads

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade
0,1,José Lima,5534900000001,Cana,Plantio,Disponível,2026-06-12 22:29:32,None,106.46
1,2,Igor Moura,5517900000002,Soja,Desenvolvimento,Disponível,NaN,None,24.76
2,3,Roberto Gomes,5514900000003,Cana,Plantio,Disponível,2026-05-31 17:49:32,None,74.21
3,4,Bruno Gomes,5566900000004,Milho,Plantio,Disponível,2026-06-14 08:51:32,None,74.73
4,5,Bruno Carvalho,5564900000005,Soja,Safra,Disponível,2026-05-30 01:13:32,None,79.37
5,6,Felipe Martins,5543900000006,Soja,Safra,Disponível,2026-06-17 08:15:32,None,81.90
6,7,José Lima,5518900000007,Cana,Entresafra,Em Cooldown,2026-06-18 19:22:32,2026-06-26 20:50:32,7.15
7,8,Eduardo Oliveira,5564900000008,Milho,Desenvolvimento,Disponível,2026-06-05 12:27:32,None,42.53
8,9,Paulo Barbosa,5534900000009,Milho,Desenvolvimento,Disponível,NaN,None,40.91
9,10,José Nascimento,5518900000010,Soja,Desenvolvimento,Em Cooldown,NaN,2026-06-27 07:50:32,9.43


In [6]:
#Verificar estrutura da amostra

amostra_leads.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_cliente        10 non-null     int64  
 1   nome              10 non-null     str    
 2   telefone          10 non-null     str    
 3   cultura           10 non-null     str    
 4   estagio_atual     10 non-null     str    
 5   status_atual      10 non-null     str    
 6   ultimo_contato    7 non-null      str    
 7   cooldown_ate      2 non-null      object 
 8   score_prioridade  10 non-null     float64
dtypes: float64(1), int64(1), object(1), str(6)
memory usage: 852.0+ bytes


In [7]:
#Analisar distribuição da amostra

amostra_leads[[
    "cultura",
    "estagio_atual",
    "status_atual",
    "score_prioridade"
]]

,cultura,estagio_atual,status_atual,score_prioridade
0,Cana,Plantio,Disponível,106.46
1,Soja,Desenvolvimento,Disponível,24.76
2,Cana,Plantio,Disponível,74.21
3,Milho,Plantio,Disponível,74.73
4,Soja,Safra,Disponível,79.37
5,Soja,Safra,Disponível,81.90
6,Cana,Entresafra,Em Cooldown,7.15
7,Milho,Desenvolvimento,Disponível,42.53
8,Milho,Desenvolvimento,Disponível,40.91
9,Soja,Desenvolvimento,Em Cooldown,9.43


In [8]:
#Criar banco SQLite

remover_banco_existente(CAMINHO_BANCO)

conexao = criar_conexao(CAMINHO_BANCO)

print("Conexão criada com sucesso.")

Conexão criada com sucesso.


In [9]:
#schema relacional

criar_schema(conexao)

Schema criado com sucesso.


In [10]:
#Inserir leads sintéticos

inserir_leads_em_lotes(
    conexao=conexao,
    quantidade_total=QUANTIDADE_LEADS,
    tamanho_lote=TAMANHO_LOTE,
    gerador=gerador
)

Gerando registros 1 até 10,000
Total inserido: 10,000
Gerando registros 10,001 até 20,000
Total inserido: 20,000
Gerando registros 20,001 até 30,000
Total inserido: 30,000
Gerando registros 30,001 até 40,000
Total inserido: 40,000
Gerando registros 40,001 até 50,000
Total inserido: 50,000

Carga concluída.
Total inserido: 50,000
Tempo total: 0.88 segundos


In [11]:
#Criar índices SQL

criar_indices(conexao)

Índices criados e banco otimizado.


In [12]:
#Validar total de leads

consulta_total = """
SELECT COUNT(*) AS total_leads
FROM leads;
"""

pd.read_sql_query(consulta_total, conexao)

,total_leads
0,50000


In [13]:
#Distribuição por status

consulta_status = """
SELECT
    status_atual,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM leads), 2) AS percentual
FROM leads
GROUP BY status_atual
ORDER BY quantidade DESC;
"""

pd.read_sql_query(consulta_status, conexao)

,status_atual,quantidade,percentual
0,Disponível,36064,72.13
1,Em Cooldown,5948,11.90
2,Convertido,3015,6.03
3,Fila Prioritária,2973,5.95
4,Em Atendimento,2000,4.00


In [14]:
#Distribuição por cultura e estágio

consulta_cultura_estagio = """
SELECT
    cultura,
    estagio_atual,
    COUNT(*) AS quantidade,
    ROUND(AVG(score_prioridade), 2) AS score_medio,
    ROUND(MIN(score_prioridade), 2) AS score_minimo,
    ROUND(MAX(score_prioridade), 2) AS score_maximo
FROM leads
GROUP BY cultura, estagio_atual
ORDER BY cultura, score_medio DESC;
"""

pd.read_sql_query(consulta_cultura_estagio, conexao)

,cultura,estagio_atual,quantidade,score_medio,score_minimo,score_maximo
0,Cana,Plantio,5533,71.28,0.0,250.00
1,Cana,Safra,5673,63.14,0.0,250.00
2,Cana,Desenvolvimento,6691,47.52,0.0,185.69
3,Cana,Entresafra,4569,33.29,0.0,130.70
4,Milho,Plantio,2549,61.12,0.0,241.55
5,Milho,Safra,2453,54.78,0.0,188.99
6,Milho,Desenvolvimento,3091,41.14,0.0,158.22
7,Milho,Entresafra,2016,28.23,0.0,119.70
8,Soja,Plantio,4423,64.55,0.0,247.39
9,Soja,Safra,4262,57.69,0.0,219.10


In [15]:
#Buscar próximos leads para robô

consulta_proximos_leads_robo = """
SELECT
    id_cliente,
    nome,
    telefone,
    cultura,
    estagio_atual,
    status_atual,
    ultimo_contato,
    cooldown_ate,
    score_prioridade
FROM leads
WHERE
    status_atual = 'Disponível'
    OR (
        status_atual = 'Em Cooldown'
        AND cooldown_ate <= datetime('now')
    )
ORDER BY score_prioridade DESC
LIMIT 20;
"""

pd.read_sql_query(consulta_proximos_leads_robo, conexao)

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade
0,16387,Ricardo Santos,5514900016387,Cana,Plantio,Disponível,2026-06-23 21:31:25,None,165.00
1,18525,Rafael Martins,5543900018525,Cana,Plantio,Disponível,2026-06-19 21:46:25,None,165.00
2,27630,Lucas Santos,5565900027630,Cana,Plantio,Disponível,2026-06-17 12:28:25,None,165.00
3,49627,André Araújo,5515900049627,Cana,Plantio,Disponível,2026-06-01 22:52:25,None,165.00
4,37706,Rafael Barbosa,5543900037706,Cana,Plantio,Disponível,2026-06-03 15:27:25,None,164.71
5,7825,Paulo Ribeiro,5516900007825,Cana,Plantio,Disponível,2026-06-17 09:36:24,None,158.91
6,7518,Roberto Moura,5511900007518,Cana,Plantio,Disponível,2026-05-29 00:04:24,None,158.73
7,37610,Fernando Costa,5514900037610,Cana,Plantio,Disponível,2026-06-18 17:49:25,None,158.36
8,23880,Fernando Rodrigues,5512900023880,Cana,Plantio,Disponível,2026-06-01 22:52:25,None,155.83
9,37545,Daniel Barbosa,5543900037545,Cana,Plantio,Disponível,NaN,None,155.59


In [16]:
#Verificar uso de índice SQL

consulta_plano_execucao = """
EXPLAIN QUERY PLAN
SELECT
    id_cliente,
    nome,
    telefone,
    cultura,
    estagio_atual,
    status_atual,
    score_prioridade
FROM leads
WHERE status_atual = 'Disponível'
ORDER BY score_prioridade DESC
LIMIT 20;
"""

pd.read_sql_query(consulta_plano_execucao, conexao)

,id,parent,notused,detail
0,5,0,156,SEARCH leads USING INDEX idx_leads_status_scor...


In [17]:
#Validar tamanho do banco

tamanho_banco_mb = CAMINHO_BANCO.stat().st_size / (1024 * 1024)

print(f"Tamanho do banco SQLite: {tamanho_banco_mb:.2f} MB")

Tamanho do banco SQLite: 11.00 MB


In [18]:
#Fechar conexão

conexao.close()

print("Conexão encerrada.")

Conexão encerrada.
